# 03 — Model Diagnostics: Bayesian Marketing Mix Model

This notebook provides a comprehensive diagnostic assessment of the trained
Bayesian MMM. All results are loaded from **pre-computed JSON artifacts**
produced during training (`models/precomputed/`).

**Sections:**

1. Setup & data loading
2. Revenue decomposition (stacked area + pie chart)
3. ROAS analysis with credible intervals
4. Response curves (saturation behaviour)
5. Adstock decay profiles
6. Parameter recovery vs ground truth
7. Budget optimisation (current vs optimal)
8. Summary & recommendations

## 1. Setup & Data Loading

In [ ]:
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 11

# Channel colour palette (consistent across all charts)
CHANNEL_COLORS = {
    "tv": "#e74c3c",
    "ooh": "#3498db",
    "print": "#2ecc71",
    "facebook": "#9b59b6",
    "search": "#f39c12",
    "base": "#95a5a6",
}
CHANNELS = ["tv", "ooh", "print", "facebook", "search"]

# --- Load all pre-computed artifacts ---
precomp = os.path.join("..", "models", "precomputed")

with open(os.path.join(precomp, "decomposition.json")) as f:
    decomp = json.load(f)
with open(os.path.join(precomp, "roas.json")) as f:
    roas_data = json.load(f)
with open(os.path.join(precomp, "response_curves.json")) as f:
    curves_data = json.load(f)
with open(os.path.join(precomp, "adstock.json")) as f:
    adstock_data = json.load(f)
with open(os.path.join(precomp, "optimal_allocation.json")) as f:
    optim_data = json.load(f)
with open(os.path.join("..", "models", "metadata.json")) as f:
    metadata = json.load(f)
with open(os.path.join("..", "data", "synthetic", "true_params.json")) as f:
    true_params = json.load(f)

print("Loaded 7 artifact files.")
print(f"Model: {metadata['model_type']}")
print(f"MCMC: {metadata['mcmc_config']['chains']} chains x {metadata['mcmc_config']['draws']} draws")
print(f"Out-of-sample MAPE: {metadata['out_of_sample']['mape']:.1f}%")

## 2. Revenue Decomposition

The decomposition breaks weekly revenue into a **base** component (trend,
seasonality, controls) and five **channel contributions** (TV, OOH, Print,
Facebook, Search). Each channel's contribution reflects the revenue
attributable to its media spend after adstock carry-over and saturation.

In [ ]:
# Build decomposition DataFrame
decomp_df = pd.DataFrame(decomp["weeks"])
decomp_df["date_week"] = pd.to_datetime(decomp_df["date_week"])

# --- Stacked area chart ---
fig, ax = plt.subplots(figsize=(14, 6))

stack_cols = ["base"] + CHANNELS
stack_data = np.array([decomp_df[col].values for col in stack_cols])
colors = [CHANNEL_COLORS[c] for c in stack_cols]
labels = [c.upper() if c != "base" else "Base" for c in stack_cols]

ax.stackplot(decomp_df["date_week"], stack_data, labels=labels, colors=colors, alpha=0.85)
ax.plot(decomp_df["date_week"], decomp_df["revenue_actual"],
        color="black", linewidth=1.2, linestyle="--", label="Actual Revenue")

ax.set_title("Weekly Revenue Decomposition by Channel", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Revenue (EUR)")
ax.legend(loc="upper left", ncol=4, fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:,.0f}k"))
plt.tight_layout()
plt.show()

In [ ]:
# --- Pie chart: total attribution ---
totals = decomp["totals"]
total_revenue = decomp_df["revenue_actual"].sum()
channel_total = sum(totals.get(ch, 0) for ch in CHANNELS)
base_total = total_revenue - channel_total

pie_labels = ["Base"] + [ch.upper() for ch in CHANNELS]
pie_values = [base_total] + [totals.get(ch, 0) for ch in CHANNELS]
pie_colors = [CHANNEL_COLORS["base"]] + [CHANNEL_COLORS[ch] for ch in CHANNELS]

fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(
    pie_values, labels=pie_labels, colors=pie_colors,
    autopct="%1.1f%%", startangle=90, pctdistance=0.75,
    textprops={"fontsize": 11},
)
for at in autotexts:
    at.set_fontsize(10)

ax.set_title("Total Revenue Attribution by Channel", fontsize=14, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()

# Print summary table
print(f"{'Component':<12} {'Total (EUR)':>14} {'Share':>8}")
print("-" * 36)
for ch in CHANNELS:
    pct = decomp["pct"].get(ch, 0)
    print(f"{ch.upper():<12} {totals.get(ch, 0):>14,.0f} {pct:>7.1f}%")
base_pct = 100 - sum(decomp["pct"].get(ch, 0) for ch in CHANNELS)
print(f"{'Base':<12} {base_total:>14,.0f} {base_pct:>7.1f}%")
print("-" * 36)
print(f"{'Total':<12} {total_revenue:>14,.0f} {'100.0':>7}%")

## 3. ROAS Analysis

**Return on Ad Spend** = total channel contribution / total channel spend.

Each channel's ROAS is computed per posterior sample, giving us a full
distribution. We report the posterior mean and 94% highest density interval
(HDI). A ROAS > 1.0 means each EUR spent returns more than 1 EUR in revenue.

In [ ]:
# ROAS bar chart with error bars (94% HDI)
roas_channels = roas_data["channels"]
ch_names = [r["channel"].upper() for r in roas_channels]
roas_means = [r["roas_mean"] for r in roas_channels]
roas_lo = [r["roas_hdi_3"] for r in roas_channels]
roas_hi = [r["roas_hdi_97"] for r in roas_channels]
bar_colors = [CHANNEL_COLORS[r["channel"]] for r in roas_channels]

yerr_lo = [roas_means[i] - roas_lo[i] for i in range(len(roas_means))]
yerr_hi = [roas_hi[i] - roas_means[i] for i in range(len(roas_means))]

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(ch_names))
bars = ax.bar(x, roas_means, color=bar_colors, alpha=0.85, edgecolor="white", linewidth=1.5)
ax.errorbar(x, roas_means, yerr=[yerr_lo, yerr_hi],
            fmt="none", ecolor="#2c3e50", capsize=6, linewidth=1.5)

# Value labels
for i, (bar, val) in enumerate(zip(bars, roas_means)):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + yerr_hi[i] + 0.01,
            f"{val:.3f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_title("Return on Ad Spend (ROAS) by Channel", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(ch_names)
ax.set_ylabel("ROAS (EUR revenue per EUR spent)")
ax.axhline(y=1.0, color="red", linestyle="--", alpha=0.4, label="Break-even (ROAS=1)")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

# Print table
print(f"{'Channel':<10} {'ROAS':>8} {'HDI 3%':>10} {'HDI 97%':>10} {'Spend':>14} {'Contribution':>14}")
print("-" * 70)
for r in roas_channels:
    print(f"{r['channel'].upper():<10} {r['roas_mean']:>8.4f} {r['roas_hdi_3']:>10.4f} "
          f"{r['roas_hdi_97']:>10.4f} {r['total_spend']:>14,.0f} {r['total_contribution']:>14,.0f}")

## 4. Response Curves (Saturation)

Response curves show the relationship between spend and contribution for
each channel. The logistic saturation function `x / (x + lambda)` creates
diminishing returns — after a threshold, additional spend yields progressively
less incremental revenue.

The vertical dashed line marks each channel's current average weekly spend.

In [ ]:
# Response curves — one subplot per channel
fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=False)

for i, ch_data in enumerate(curves_data["channels"]):
    ax = axes[i]
    ch = ch_data["channel"]
    spends = [pt["spend"] for pt in ch_data["curve"]]
    contribs = [pt["contribution"] for pt in ch_data["curve"]]

    ax.plot(spends, contribs, color=CHANNEL_COLORS[ch], linewidth=2.5)
    ax.fill_between(spends, 0, contribs, color=CHANNEL_COLORS[ch], alpha=0.15)

    # Current average spend marker
    current_spend = optim_data["current"].get(ch, 0)
    if current_spend > 0:
        ax.axvline(x=current_spend, color="#2c3e50", linestyle="--", alpha=0.6, linewidth=1.2)
        ax.set_xlabel(f"Avg: {current_spend/1000:.1f}k", fontsize=9, color="#555")

    ax.set_title(ch.upper(), fontsize=12, fontweight="bold", color=CHANNEL_COLORS[ch])
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))

axes[0].set_ylabel("Contribution (scaled)")
fig.suptitle("Response Curves: Spend vs Contribution", fontsize=14, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

## 5. Adstock Decay Profiles

Geometric adstock models the **carry-over** effect of advertising. An ad
shown in week 0 still generates some impact in weeks 1, 2, ... up to `l_max`.

The decay rate **alpha** determines how quickly the effect fades:
- alpha ~ 0 → impact is immediate, no carry-over (e.g. search)
- alpha ~ 1 → impact persists for many weeks (e.g. TV brand campaigns)

In [ ]:
# Adstock decay bar charts
fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=True)

for i, ch_data in enumerate(adstock_data["channels"]):
    ax = axes[i]
    ch = ch_data["channel"]
    decay = ch_data["decay_vector"]
    weeks = list(range(len(decay)))

    ax.bar(weeks, decay, color=CHANNEL_COLORS[ch], alpha=0.85, edgecolor="white")
    ax.set_title(f"{ch.upper()}\nalpha={ch_data['alpha_mean']:.3f}",
                 fontsize=11, fontweight="bold", color=CHANNEL_COLORS[ch])
    ax.set_xlabel("Lag (weeks)")
    ax.set_xticks(weeks)

axes[0].set_ylabel("Weight")
fig.suptitle("Adstock Decay Profiles (Geometric)", fontsize=14, fontweight="bold", y=1.05)
plt.tight_layout()
plt.show()

# Print alpha estimates with HDI
print(f"{'Channel':<10} {'Alpha':>8} {'HDI 3%':>10} {'HDI 97%':>10}")
print("-" * 42)
for ch_data in adstock_data["channels"]:
    print(f"{ch_data['channel'].upper():<10} {ch_data['alpha_mean']:>8.4f} "
          f"{ch_data['alpha_hdi_3']:>10.4f} {ch_data['alpha_hdi_97']:>10.4f}")

## 6. Parameter Recovery

Since the model was trained on **synthetic data with known true parameters**,
we can validate the model by checking whether the posterior distributions
contain the true values. A parameter is "recovered" if the true value falls
within the 94% HDI of its posterior.

**Result: 9/10 parameters recovered.** Only TV adstock alpha was missed.

In [ ]:
# Parameter recovery: True vs Estimated grouped bar charts
recovery = metadata["parameter_recovery"]
channels = list(recovery["adstock_alphas"].keys())

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
x = np.arange(len(channels))
width = 0.35

# --- Adstock Alphas ---
ax = axes[0]
true_a = [recovery["adstock_alphas"][ch]["true"] for ch in channels]
est_a = [recovery["adstock_alphas"][ch]["estimated"] for ch in channels]
hdi_lo_a = [recovery["adstock_alphas"][ch]["hdi_3"] for ch in channels]
hdi_hi_a = [recovery["adstock_alphas"][ch]["hdi_97"] for ch in channels]
in_hdi_a = [recovery["adstock_alphas"][ch]["in_hdi"] for ch in channels]

bars_t = ax.bar(x - width/2, true_a, width, label="True", color="#2c3e50", alpha=0.8)
bars_e = ax.bar(x + width/2, est_a, width, label="Estimated", color="#3498db", alpha=0.8)
yerr = [[est_a[i] - hdi_lo_a[i] for i in range(len(channels))],
        [hdi_hi_a[i] - est_a[i] for i in range(len(channels))]]
ax.errorbar(x + width/2, est_a, yerr=yerr, fmt="none", ecolor="black", capsize=4)

for i, ok in enumerate(in_hdi_a):
    if not ok:
        bars_t[i].set_edgecolor("red")
        bars_t[i].set_linewidth(2.5)
        ax.annotate("MISS", xy=(x[i], true_a[i] + 0.05),
                    ha="center", fontsize=9, color="red", fontweight="bold")

ax.set_title("Adstock Alpha: True vs Estimated", fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels([ch.upper() for ch in channels])
ax.set_ylabel("Alpha")
ax.legend()
ax.set_ylim(0, 1.0)

# --- Saturation Lambdas ---
ax = axes[1]
true_l = [recovery["saturation_lambdas"][ch]["true"] for ch in channels]
est_l = [recovery["saturation_lambdas"][ch]["estimated"] for ch in channels]
hdi_lo_l = [recovery["saturation_lambdas"][ch]["hdi_3"] for ch in channels]
hdi_hi_l = [recovery["saturation_lambdas"][ch]["hdi_97"] for ch in channels]
in_hdi_l = [recovery["saturation_lambdas"][ch]["in_hdi"] for ch in channels]

ax.bar(x - width/2, true_l, width, label="True", color="#2c3e50", alpha=0.8)
ax.bar(x + width/2, est_l, width, label="Estimated", color="#e67e22", alpha=0.8)
yerr_l = [[est_l[i] - hdi_lo_l[i] for i in range(len(channels))],
          [hdi_hi_l[i] - est_l[i] for i in range(len(channels))]]
ax.errorbar(x + width/2, est_l, yerr=yerr_l, fmt="none", ecolor="black", capsize=4)

for i, ok in enumerate(in_hdi_l):
    if not ok:
        ax.annotate("MISS", xy=(x[i], true_l[i] + 0.3),
                    ha="center", fontsize=9, color="red", fontweight="bold")

ax.set_title("Saturation Lambda: True vs Estimated", fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels([ch.upper() for ch in channels])
ax.set_ylabel("Lambda")
ax.legend()

plt.tight_layout()
plt.show()

# Summary table
n_ok = sum(1 for v in recovery["adstock_alphas"].values() if v["in_hdi"])
n_ok += sum(1 for v in recovery["saturation_lambdas"].values() if v["in_hdi"])
n_total = len(recovery["adstock_alphas"]) + len(recovery["saturation_lambdas"])
print(f"\nRecovered: {n_ok}/{n_total} parameters within 94% HDI")

## 7. Budget Optimisation

Given the same total weekly budget, the optimizer reallocates spend across
channels to maximise predicted revenue. The comparison shows current
(historical average) vs optimal allocation, along with per-channel
recommendations (increase, decrease, or maintain).

In [ ]:
# Budget optimisation: current vs optimal side-by-side
current = optim_data["current"]
optimal = optim_data["optimal"]

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(CHANNELS))
width = 0.35

curr_vals = [current[ch] for ch in CHANNELS]
opt_vals = [optimal[ch] for ch in CHANNELS]

bars_c = ax.bar(x - width/2, curr_vals, width, label="Current", color="#95a5a6", alpha=0.85)
bars_o = ax.bar(x + width/2, opt_vals, width,
                label="Optimal", color=[CHANNEL_COLORS[ch] for ch in CHANNELS], alpha=0.85)

ax.set_title("Weekly Budget Allocation: Current vs Optimal", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels([ch.upper() for ch in CHANNELS])
ax.set_ylabel("Weekly Spend (EUR)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:,.1f}k"))
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

# KPI cards
print(f"Total weekly budget: EUR {optim_data['total_budget']:,.0f}")
print(f"Current revenue:     EUR {optim_data['current_revenue']:,.0f}")
print(f"Optimal revenue:     EUR {optim_data['optimal_revenue']:,.0f}")
print(f"Lift:                EUR {optim_data['lift_abs']:,.0f} ({optim_data['lift_pct']:.1f}%)")
print()

# Recommendations table
print(f"{'Channel':<10} {'Current':>10} {'Optimal':>10} {'Delta':>10} {'Action':<10}")
print("-" * 55)
for rec in optim_data["recommendations"]:
    print(f"{rec['channel'].upper():<10} {rec['current']:>10,.0f} {rec['optimal']:>10,.0f} "
          f"{rec['delta']:>+10,.0f} {rec['action']:<10}")

## 8. Summary

### Model Quality

| Metric | Value | Status |
|--------|-------|--------|
| Max R-hat | 1.007 | PASSED (< 1.01) |
| Min ESS bulk | 2,032 | PASSED (> 400) |
| Min ESS tail | 1,384 | PASSED (> 400) |
| Out-of-sample MAPE | 3.9% | PASSED (< 15%) |
| Out-of-sample R2 | 0.91 | Excellent |
| Parameter recovery | 9/10 | Good |

### Key Findings

1. **Base dominates** — the majority of revenue comes from organic demand
   (trend, seasonality, brand equity), not incremental media spend.
2. **Print has the highest ROAS** (0.28), followed by Search (0.20) and TV (0.17).
3. **All channels show sub-1.0 ROAS** — this is typical for MMMs that attribute
   revenue in absolute EUR terms rather than marginal lift.
4. **Adstock alphas are moderate** across all channels, with wide posteriors
   reflecting the difficulty of separating carry-over from saturation effects.
5. **TV adstock alpha was the only parameter not recovered** — a known
   identifiability challenge where adstock and saturation trade off.

### Next Steps

- **Session C (App Building):** FastAPI serves these pre-computed JSONs,
  React dashboard provides interactive decomposition, ROAS, simulator, and
  budget optimisation views.
- **Session D (Documentation):** Model card, full documentation, executive deck.